In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_PATH = Path("../data/raw/online_retail_II.xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Data file not found at {DATA_PATH}. Please ensure the file exists.")

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("File location:", DATA_PATH.resolve())
print(f"File size: {DATA_PATH.stat().st_size / (1024**2):.2f} MB")

Pandas version: 3.0.6
NumPy version: 2.4.6
File location: C:\Users\SAMSUNG\OneDrive\Desktop\PythonProjects\retail-pulse-analytics\data\raw\online_retail_II.xlsx
File size: 43.51 MB


In [3]:
with pd.ExcelFile(DATA_PATH, engine="openpyxl") as workbook:
    sheet_names = workbook.sheet_names

print("Sheet names:", sheet_names)

Sheet names: ['Year 2009-2010', 'Year 2010-2011']


In [5]:
df = pd.read_excel(DATA_PATH, sheet_name=sheet_names[0], engine="openpyxl")

print("Loaded worksheet:", sheet_names[0])
print("DataFrame shape:", df.shape)

Loaded worksheet: Year 2009-2010
DataFrame shape: (525461, 8)


In [6]:
df = pd.read_excel(
    DATA_PATH,
    sheet_name=None,
    engine="openpyxl"
)

df = pd.concat(
    df.values(),
    ignore_index=True
)

print("Combined dataset shape:", df.shape)

Combined dataset shape: (1067371, 8)


In [19]:
duplicate_mask = df.duplicated(keep=False)

duplicate_profile = (
    df.loc[duplicate_mask]
      .value_counts()
      .rename("occurrences")
      .reset_index()
      .sort_values("occurrences", ascending=False)
      
)

display(duplicate_profile.head(10))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,occurrences
0,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,20
1,555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,12
2,537224,70007,HI TEC ALPINE HAND WARMER,1,2010-12-05 16:24:00,1.65,13174.0,United Kingdom,10
3,572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,2011-10-26 12:46:00,1.25,14102.0,United Kingdom,8
4,496431,84826,ASSTD DESIGN 3D PAPER STICKERS,1,2010-02-01 12:30:00,0.85,16415.0,United Kingdom,6
5,502660,17021,NAMASTE SWAGAT INCENSE,6,2010-03-25 17:18:00,0.30,13187.0,United Kingdom,6
6,525065,20894,HANGING BAUBLE T-LIGHT HOLDER LARGE,1,2010-10-03 14:28:00,2.95,16799.0,United Kingdom,6
7,534219,35953,FOLKART STAR CHRISTMAS DECORATIONS,1,2010-11-21 16:13:00,1.25,13230.0,United Kingdom,6
8,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom,6
9,536464,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 12:23:00,2.10,17968.0,United Kingdom,6


In [20]:
df_clean = df.copy()

rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates(keep="first").copy()

rows_after = len(df_clean)

rows_removed = rows_before - rows_after

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_removed)
print("Remaining exact duplicates:", df_clean.duplicated().sum())

Rows before: 1067371
Rows after: 1033036
Rows removed: 34335
Remaining exact duplicates: 0


In [26]:
duplicate_audit = {
    "rows_before": rows_before,
    "rows_removed": rows_removed,
    "rows_after": rows_after,
    "removal_percentage": round((rows_removed / rows_before) * 100, 2),
}

display(pd.Series(duplicate_audit, name="value"))

rows_before           1067371.00
rows_removed            34335.00
rows_after            1033036.00
removal_percentage          3.22
Name: value, dtype: float64